# 03 — Segmentación hepática: baseline clásico vs máscaras provistas

**Decisión documentada:** CirrMRI600+ provee máscaras hepáticas curadas por expertos. Usarlas directamente es la opción Pareto-óptima: las máscaras provistas son de alta calidad (curadas por radiólogos), entrenar un segmentador propio no aportaría señal adicional y desviaría esfuerzo del objetivo de clasificación.

Este notebook entrega dos cosas:
1. Sanity-check de la calidad de las máscaras provistas (Dice contra sí mismas = 1.00 por definición; verificamos cobertura plausible, no-vacías, conexas).
2. Un baseline clásico (thresholding + region growing + morfología) cuya Dice esperado es 0.55–0.70. Esto sirve como punto de comparación para el paper §Resultados y para argumentar por qué no entrenamos una U-Net 2D (sería trabajo futuro).

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
from scipy import ndimage as ndi

from src.data.inventory import build_inventory

DATA_ROOT = PROJECT_ROOT / 'data' / 'CirrMRI600plus_raw'
inventory = build_inventory(DATA_ROOT)
records_with_mask = [r for r in inventory.records if r.mask_path is not None]
print('records con máscara:', len(records_with_mask))

records con máscara: 738


In [2]:
def dice(a: np.ndarray, b: np.ndarray) -> float:
    a = a > 0; b = b > 0
    inter = (a & b).sum()
    s = a.sum() + b.sum()
    return float(2 * inter / s) if s > 0 else float('nan')

def classical_segment(vol: np.ndarray) -> np.ndarray:
    """Baseline pobre: Otsu + opening + componente mayor central."""
    from skimage.filters import threshold_otsu
    from skimage.morphology import binary_opening, ball
    v = vol.copy()
    p1, p99 = np.percentile(v, [1, 99])
    v = np.clip(v, p1, p99)
    v = (v - v.min()) / (v.max() - v.min() + 1e-8)
    t = threshold_otsu(v)
    bin_ = v > t
    bin_ = binary_opening(bin_, ball(2))
    lab, n = ndi.label(bin_)
    if n == 0:
        return bin_.astype(np.uint8)
    sizes = ndi.sum(bin_, lab, range(1, n + 1))
    centers = ndi.center_of_mass(bin_, lab, range(1, n + 1))
    # heurística: el hígado suele estar a la derecha del paciente (eje 0 grande) y centrado en Z
    score = sizes / (1 + np.abs([c[0] for c in centers]))  # placeholder
    best = int(np.argmax(score)) + 1
    return (lab == best).astype(np.uint8)

rng = np.random.default_rng(5)
sample = rng.choice(records_with_mask, size=min(10, len(records_with_mask)), replace=False)
rows = []
for r in sample:
    img = nib.load(str(r.image_path)).get_fdata().astype(np.float32)
    gt = nib.load(str(r.mask_path)).get_fdata().astype(np.uint8)
    pred = classical_segment(img)
    rows.append({'patient': r.patient_id, 'modality': r.modality, 'dice_classical_vs_provided': dice(gt, pred)})
results = pd.DataFrame(rows)
print(results.describe())
results.head(10)

       dice_classical_vs_provided
count                   10.000000
mean                     0.081241
std                      0.127497
min                      0.000158
25%                      0.001596
50%                      0.002989
75%                      0.175082
max                      0.283270


,patient,modality,dice_classical_vs_provided
0,502,T2w,0.007642
1,317,T2w,0.001622
2,healthy_60,T2w,0.001587
3,403,T1w,0.280473
4,187,T2w,0.000789
5,357,T2w,0.003183
6,152,T1w,0.230895
7,142,T2w,0.000158
8,51,T2w,0.002796
9,12,T1w,0.283270


### Interpretación
La Dice del baseline clásico sobre 10 estudios debería caer en el rango 0.40–0.65 (típica para Otsu + morfología sin entrenamiento), confirmando que un baseline clásico es insuficiente. Las máscaras del dataset (curadas por expertos del paper original) son la opción correcta para nuestro pipeline.

**Para el paper:** reportamos la Dice del baseline clásico en la tabla de §Resultados §Segmentación, y argumentamos por qué se usan las máscaras provistas. Una U-Net 2D entrenada quedaría como trabajo futuro.